In [1]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "../data/processed/cleaned_data.csv"

print("Dataset path:")
print(os.path.abspath(DATA_PATH))

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Shape:", df.shape)

Dataset path:
f:\NIDS-ML-NEW\data\processed\cleaned_data.csv

Dataset loaded successfully!
Shape: (2830743, 79)


In [2]:
print("Target column:", df["Label"].name)

print("\nNumber of classes:", df["Label"].nunique())

print("\nClasses:")
print(df["Label"].unique())

Target column: Label

Number of classes: 15

Classes:
<ArrowStringArray>
[                    'BENIGN',                       'DDoS',
                   'PortScan',                        'Bot',
               'Infiltration',   'Web Attack - Brute Force',
           'Web Attack - XSS', 'Web Attack - Sql Injection',
                'FTP-Patator',                'SSH-Patator',
              'DoS slowloris',           'DoS Slowhttptest',
                   'DoS Hulk',              'DoS GoldenEye',
                 'Heartbleed']
Length: 15, dtype: str


In [3]:
X = df.drop(columns=["Label"])
y = df["Label"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (2830743, 78)
Target shape: (2830743,)


In [4]:
unique_counts = X.nunique()

constant_features = unique_counts[
    unique_counts == 1
].index.tolist()

print("Constant features:")
for feature in constant_features:
    print("-", feature)

print("\nNumber of constant features:", len(constant_features))

Constant features:
- Bwd PSH Flags
- Bwd URG Flags
- Fwd Avg Bytes/Bulk
- Fwd Avg Packets/Bulk
- Fwd Avg Bulk Rate
- Bwd Avg Bytes/Bulk
- Bwd Avg Packets/Bulk
- Bwd Avg Bulk Rate

Number of constant features: 8


In [5]:
X = X.drop(columns=constant_features)

print("Shape after removing constant features:")
print(X.shape)

Shape after removing constant features:
(2830743, 70)


In [6]:
correlation_sample = X.sample(
    n=10000,
    random_state=42
)

correlation_matrix = correlation_sample.corr()

print("Correlation matrix shape:")
print(correlation_matrix.shape)

Correlation matrix shape:
(70, 70)


In [7]:
upper = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

correlated_features = [
    column
    for column in upper.columns
    if any(upper[column].abs() > 0.95)
]

print("Highly correlated features to remove:")
for feature in correlated_features:
    print("-", feature)

print(
    "\nNumber of features selected for removal:",
    len(correlated_features)
)

Highly correlated features to remove:
- Total Backward Packets
- Total Length of Bwd Packets
- Fwd Packet Length Std
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Fwd IAT Total
- Fwd IAT Max
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Packet Length Std
- SYN Flag Count
- CWE Flag Count
- Average Packet Size
- Avg Fwd Segment Size
- Avg Bwd Segment Size
- Fwd Header Length.1
- Subflow Fwd Packets
- Subflow Fwd Bytes
- Subflow Bwd Packets
- Subflow Bwd Bytes
- act_data_pkt_fwd
- Idle Mean
- Idle Max
- Idle Min

Number of features selected for removal: 25


In [8]:
X = X.drop(columns=correlated_features)

print("Shape after removing highly correlated features:")
print(X.shape)

Shape after removing highly correlated features:
(2830743, 45)


In [9]:
print("Remaining features:")
print()

for i, feature in enumerate(X.columns, start=1):
    print(f"{i:2}. {feature}")

print("\nTotal remaining features:", X.shape[1])

Remaining features:

 1. Destination Port
 2. Flow Duration
 3. Total Fwd Packets
 4. Total Length of Fwd Packets
 5. Fwd Packet Length Max
 6. Fwd Packet Length Min
 7. Fwd Packet Length Mean
 8. Bwd Packet Length Max
 9. Bwd Packet Length Min
10. Flow Bytes/s
11. Flow Packets/s
12. Flow IAT Mean
13. Flow IAT Std
14. Flow IAT Max
15. Flow IAT Min
16. Fwd IAT Mean
17. Fwd IAT Std
18. Fwd IAT Min
19. Bwd IAT Total
20. Bwd IAT Mean
21. Bwd IAT Std
22. Bwd IAT Max
23. Bwd IAT Min
24. Fwd PSH Flags
25. Fwd URG Flags
26. Bwd Packets/s
27. Min Packet Length
28. Max Packet Length
29. Packet Length Mean
30. Packet Length Variance
31. FIN Flag Count
32. RST Flag Count
33. PSH Flag Count
34. ACK Flag Count
35. URG Flag Count
36. ECE Flag Count
37. Down/Up Ratio
38. Init_Win_bytes_forward
39. Init_Win_bytes_backward
40. min_seg_size_forward
41. Active Mean
42. Active Std
43. Active Max
44. Active Min
45. Idle Std

Total remaining features: 45


In [10]:
print("Missing values:", X.isnull().sum().sum())

print(
    "Infinite values:",
    np.isinf(X.select_dtypes(include=np.number)).sum().sum()
)

Missing values: 0
Infinite values: 0


In [11]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Original classes:")
print(label_encoder.classes_)

print("\nEncoded classes:")
for original, encoded in zip(
    label_encoder.classes_,
    range(len(label_encoder.classes_))
):
    print(f"{original} -> {encoded}")

Original classes:
['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack - Brute Force' 'Web Attack - Sql Injection'
 'Web Attack - XSS']

Encoded classes:
BENIGN -> 0
Bot -> 1
DDoS -> 2
DoS GoldenEye -> 3
DoS Hulk -> 4
DoS Slowhttptest -> 5
DoS slowloris -> 6
FTP-Patator -> 7
Heartbleed -> 8
Infiltration -> 9
PortScan -> 10
SSH-Patator -> 11
Web Attack - Brute Force -> 12
Web Attack - Sql Injection -> 13
Web Attack - XSS -> 14


In [12]:
print("\nEncoded target shape:", y_encoded.shape)
print("Target data type:", y_encoded.dtype)
print("Number of classes:", len(label_encoder.classes_))


Encoded target shape: (2830743,)
Target data type: int64
Number of classes: 15


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTemporary set:")
print("X_temp:", X_temp.shape)
print("y_temp:", y_temp.shape)

Training set:
X_train: (1981520, 45)
y_train: (1981520,)

Temporary set:
X_temp: (849223, 45)
y_temp: (849223,)


In [14]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation set:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTest set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Training set:
X_train: (1981520, 45)
y_train: (1981520,)

Validation set:
X_val: (424611, 45)
y_val: (424611,)

Test set:
X_test: (424612, 45)
y_test: (424612,)


In [15]:
print("Training class distribution:")
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

print("\nValidation class distribution:")
print(pd.Series(y_val).value_counts(normalize=True).sort_index())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts(normalize=True).sort_index())

Training class distribution:
0     0.803004
1     0.000694
2     0.045227
3     0.003636
4     0.081630
5     0.001942
6     0.002047
7     0.002804
8     0.000004
9     0.000013
10    0.056144
11    0.002083
12    0.000532
13    0.000008
14    0.000230
Name: proportion, dtype: float64

Validation class distribution:
0     0.803003
1     0.000695
2     0.045227
3     0.003636
4     0.081630
5     0.001943
6     0.002049
7     0.002803
8     0.000005
9     0.000014
10    0.056143
11    0.002082
12    0.000532
13    0.000007
14    0.000231
Name: proportion, dtype: float64

Test class distribution:
0     0.803004
1     0.000695
2     0.045227
3     0.003636
4     0.081630
5     0.001943
6     0.002047
7     0.002805
8     0.000002
9     0.000012
10    0.056145
11    0.002084
12    0.000532
13    0.000007
14    0.000231
Name: proportion, dtype: float64


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

print("Training scaled shape:", X_train_scaled.shape)
print("Validation scaled shape:", X_val_scaled.shape)
print("Test scaled shape:", X_test_scaled.shape)

Training scaled shape: (1981520, 45)
Validation scaled shape: (424611, 45)
Test scaled shape: (424612, 45)


In [17]:
print("Training mean:", X_train_scaled.mean())
print("Training standard deviation:", X_train_scaled.std())

Training mean: -5.2579671583648975e-18
Training standard deviation: 1.0


In [18]:
import os
import joblib

PREPROCESSING_DIR = "../models/preprocessing"

os.makedirs(PREPROCESSING_DIR, exist_ok=True)

print("Preprocessing directory:")
print(os.path.abspath(PREPROCESSING_DIR))

Preprocessing directory:
f:\NIDS-ML-NEW\models\preprocessing


In [19]:
joblib.dump(
    scaler,
    os.path.join(PREPROCESSING_DIR, "scaler.joblib")
)

joblib.dump(
    label_encoder,
    os.path.join(PREPROCESSING_DIR, "label_encoder.joblib")
)

print("Preprocessing objects saved successfully!")

Preprocessing objects saved successfully!


In [20]:
print("Files in preprocessing directory:")

for file in os.listdir(PREPROCESSING_DIR):
    print("-", file)

Files in preprocessing directory:
- .gitkeep
- label_encoder.joblib
- scaler.joblib


In [21]:
print("=" * 60)
print("FEATURE ENGINEERING FINAL VERIFICATION")
print("=" * 60)

print("\nOriginal dataset:")
print("Rows:", df.shape[0])
print("Original features:", df.shape[1] - 1)

print("\nAfter feature selection:")
print("Remaining features:", X.shape[1])

print("\nDataset splits:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nScaled datasets:")
print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

print("\nTarget:")
print("Number of classes:", len(label_encoder.classes_))
print("Encoded target type:", y_encoded.dtype)

print("\nData quality:")
print("Training NaN:", np.isnan(X_train_scaled).sum())
print("Validation NaN:", np.isnan(X_val_scaled).sum())
print("Test NaN:", np.isnan(X_test_scaled).sum())

print("\nPreprocessing:")
print("Scaler:", type(scaler).__name__)
print("Label encoder:", type(label_encoder).__name__)

print("\n" + "=" * 60)
print("FEATURE ENGINEERING COMPLETED SUCCESSFULLY!")
print("=" * 60)

FEATURE ENGINEERING FINAL VERIFICATION

Original dataset:
Rows: 2830743
Original features: 78

After feature selection:
Remaining features: 45

Dataset splits:
Training: (1981520, 45)
Validation: (424611, 45)
Test: (424612, 45)

Scaled datasets:
Training: (1981520, 45)
Validation: (424611, 45)
Test: (424612, 45)

Target:
Number of classes: 15
Encoded target type: int64

Data quality:
Training NaN: 0
Validation NaN: 0
Test NaN: 0

Preprocessing:
Scaler: StandardScaler
Label encoder: LabelEncoder

FEATURE ENGINEERING COMPLETED SUCCESSFULLY!
